# Persisting Indexes & Production Vector Stores

Every episode so far has rebuilt its `VectorStoreIndex` from scratch, in memory, every single run — great for a self-contained tutorial notebook, terrible for a real app that would otherwise re-embed the same documents (and re-pay for those embedding calls) on every restart.

This episode covers two ways to fix that:

1. **Persist the default in-memory store to disk** with `storage_context.persist()` / `load_index_from_storage()` — zero new dependencies, just files on disk.
2. **Swap in a purpose-built embedded vector database** (Chroma) that persists automatically as you write to it — closer to what a real production RAG app would use, still with zero external servers or API keys.


**Step 1 — Set up and clean the slate.** Configures the usual LLM/embedding models, then deletes any `storage_demo/`/`chroma_demo/` folders left over from a previous run so this notebook produces the same result every time it's rerun from the top.


In [1]:
import logging
import shutil

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from the HTTP client and LlamaIndex itself.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Loads .env into os.environ so OPENAI_API_KEY is available to the clients below.
load_dotenv()

Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Delete any storage/Chroma folders left over from a previous run — this keeps
# the notebook idempotent so re-running it from the top always starts fresh.
shutil.rmtree("storage_demo", ignore_errors=True)
shutil.rmtree("chroma_demo", ignore_errors=True)

**Step 2 — Build an index, then persist it to disk.** `storage_context.persist()` writes the docstore, vector store, and index metadata out as JSON files — state that would otherwise vanish the moment the Python process exits.


In [2]:
from pathlib import Path

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)

# Writes the docstore, vector store, and index metadata to disk as JSON files —
# this is the state that would otherwise be lost when the process exits.
index.storage_context.persist(persist_dir="storage_demo")
written_files = sorted(p.name for p in Path("storage_demo").iterdir())
print(f"Persisted {len(index.docstore.docs)} nodes to storage_demo/")
print(f"Files written: {written_files}")

Persisted 8 nodes to storage_demo/
Files written: ['default__vector_store.json', 'docstore.json', 'graph_store.json', 'image__vector_store.json', 'index_store.json']


**Step 3 — Reload the index from disk.** `load_index_from_storage()` rebuilds the exact same index from the persisted JSON files, with zero new embedding API calls — proving the persisted state is actually complete and reusable.


In [3]:
from llama_index.core import StorageContext, load_index_from_storage

# A fresh StorageContext pointed at the same directory reloads the exact same
# nodes and embeddings — no OpenAI embedding calls happen on this reload.
reloaded_storage_context = StorageContext.from_defaults(persist_dir="storage_demo")
reloaded_index = load_index_from_storage(reloaded_storage_context)

response = reloaded_index.as_query_engine().query("What is Naruto's signature technique?")
print(f"Reloaded index has {len(reloaded_index.docstore.docs)} nodes")
print(response)

Reloaded index has 8 nodes
Naruto's signature technique is the Rasengan, a swirling ball of concentrated chakra.


## A real embedded vector database: Chroma

The approach above persists LlamaIndex's own default `SimpleVectorStore` — fine for a demo, but it re-loads the _entire_ store into memory at once, which doesn't scale past a certain size. Swapping in a dedicated vector database backend like Chroma changes none of the query code, only how the storage layer works underneath.


**Step 4 — Build a Chroma-backed index.** Creating a `PersistentClient` + collection, then wrapping it in a `ChromaVectorStore` and passing that into a `StorageContext`, swaps the storage backend from LlamaIndex's default in-memory store to Chroma — the `from_documents()` call itself doesn't change.


In [4]:
import chromadb
from llama_index.core import StorageContext as ChromaStorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

# A PersistentClient writes to disk at the given path automatically, unlike
# Chroma's default in-memory client.
chroma_client = chromadb.PersistentClient(path="chroma_demo")
chroma_collection = chroma_client.get_or_create_collection("anime_docs")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)  # wraps Chroma as a LlamaIndex vector store
storage_context = ChromaStorageContext.from_defaults(vector_store=vector_store)

# Same from_documents() call as before — only the storage_context is different.
chroma_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(f"Chroma collection now holds {chroma_collection.count()} vectors, written to chroma_demo/")

Chroma collection now holds 8 vectors, written to chroma_demo/


**Step 5 — Simulate reopening in a brand-new process.** A fresh `PersistentClient` pointed at the same `chroma_demo/` path, wrapped with `VectorStoreIndex.from_vector_store()`, immediately sees all the previously written vectors — no documents or re-embedding required.


In [6]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

# Simulate a fresh process: a brand-new client pointed at the same path sees the
# same data immediately — Chroma wrote it to disk as we indexed, no explicit
# persist() call needed.
reopened_client = chromadb.PersistentClient(path="chroma_demo")
reopened_collection = reopened_client.get_or_create_collection("anime_docs")
reopened_store = ChromaVectorStore(chroma_collection=reopened_collection)
reopened_index = VectorStoreIndex.from_vector_store(
    reopened_store
)  # builds an index directly from an existing vector store, no documents needed

response = reopened_index.as_query_engine().query("How many Dragon Balls are needed to summon Shenron?")
print(f"Reopened Chroma collection has {reopened_collection.count()} vectors")
print(response)

Reopened Chroma collection has 8 vectors
Seven Dragon Balls are needed to summon Shenron.


### Summary

- The default `SimpleVectorStore` is in-memory only — persist it explicitly with `storage_context.persist()` and reload with `load_index_from_storage()`, or you re-embed everything on every run.
- Swapping to a real vector database like Chroma is a storage-layer change only — `VectorStoreIndex.from_documents()` and every query engine built on top of it work identically regardless of which store is underneath.
- `storage_demo/` and `chroma_demo/` are git-ignored — they're regenerated artifacts, not something to commit.
